# CMSC 173 &middot; Machine Learning &mdash; Week 3 Lab
## Linear Regression: Fitting a Line by Gradient Descent

Last week you *estimated* parameters. This week you **fit a model**: a straight line through
data. You will build the two engines that do it &mdash; **gradient descent** (take small
downhill steps) and the **normal equation** (solve it in one shot) &mdash; entirely from
scratch.

**We use the lecture's data.** The five condo units from the slides: floor area against
price. Every number you print here should match what you saw in class, so you can tell
immediately when something has gone wrong. Then, at the end, the *same code* runs on
27,601 real government contracts.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib only (no scikit-learn until week 4).** **Not graded.** About 55 minutes.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
print("ready")

---
### Part 1 &middot; The data from the lecture

Five condo units. Floor area in square metres, price in millions of pesos.

| unit | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| area (sqm) | 20 | 40 | 60 | 80 | 100 |
| price (&#8369;M) | 2 | 4 | 5 | 4 | 5 |

Notice unit 4: 80 sqm, but **cheaper** than the 60 sqm unit. Location, age and floor level
are not in this table, and they matter. That is why no straight line will pass through all
five points &mdash; and why we need a definition of "best".

In [ ]:
area  = np.array([20, 40, 60, 80, 100], dtype=float)   # (1) the feature, in sqm
price = np.array([ 2,  4,  5,  4,   5], dtype=float)   # (2) the target, in PHP millions
m = len(area)                                          # (3) how many observations

print("area  =", area)
print("price =", price)
print("mean area  =", area.mean(), "sqm")              # (4) should print 60.0
print("mean price =", price.mean(), "million")         # (5) should print 4.0

plt.figure(figsize=(6, 3.2))                           # (6)
plt.scatter(area, price, s=70, zorder=3)
plt.xlabel("Floor area (sqm)"); plt.ylabel("Price (PHP millions)")
plt.grid(alpha=.3); plt.title("No single line hits all five")
plt.show()

**Reading the code, line by line:**

**Answer here:**

Line (4) prints the mean area and line (5) the mean price. In the lecture we proved the
fitted line must pass through $(\bar{x}, \bar{y})$.

**What point must our line go through?** Write it down now &mdash; you will check it in Part 3.



---
### Part 2 &middot; The cost of a guess

A *guess* is a pair $(\theta_0, \theta_1)$: a base price and a price per square metre.
The **cost** $J$ measures how wrong that guess is, averaged over all five units:

$$J(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^{m}\left(\hat{y}^{(i)} - y^{(i)}\right)^2$$

Guess &#8369;1M base plus &#8369;50,000 per sqm and the lecture got $J = 0.40$. Let us
reproduce that.

In [ ]:
def predict(area, b0, b1):          # (1) the line's prediction for each unit
    return b0 + b1 * area

def cost(area, price, b0, b1):      # (2) mean squared error, halved
    err = predict(area, b0, b1) - price
    return (err ** 2).sum() / (2 * len(area))

for b0, b1 in [(1.0, 0.05), (2.0, 0.035), (2.2, 0.03), (3.0, 0.015)]:   # (3)
    print(f"base {b0:4.2f}M  per-sqm {b1:.3f}M   J = {cost(area, price, b0, b1):.4f}")

**Reading the code, line by line:**

**Answer here:**

Line (3) tries four guesses. One of them is the answer the lecture derived.

**Which pair gives the smallest $J$, and what does its $\theta_1$ mean in pesos per square
metre?**



---
### Part 3 &middot; Solve it exactly (the normal equation)

No searching. The lecture derived a closed form:

$$\theta_1 = \frac{\sum(x-\bar{x})(y-\bar{y})}{\sum(x-\bar{x})^2}, \qquad
\theta_0 = \bar{y} - \theta_1\bar{x}$$

In [ ]:
dx = area  - area.mean()                    # (1) deviations from the mean area
dy = price - price.mean()                   # (2) deviations from the mean price

Sxy = (dx * dy).sum()                       # (3) how area and price vary TOGETHER
Sxx = (dx ** 2).sum()                       # (4) how area varies alone

theta1 = Sxy / Sxx                          # (5) the slope
theta0 = price.mean() - theta1 * area.mean()  # (6) the intercept

print("Sxy =", Sxy, "   Sxx =", Sxx)        # (7) lecture said 120 and 4000
print(f"theta1 = {theta1}  PHP M per sqm  =  PHP {theta1*1e6:,.0f} per sqm")
print(f"theta0 = {theta0}  PHP M")
print("passes through (60, 4)?", np.isclose(predict(60, theta0, theta1), 4.0))   # (8)

**Reading the code, line by line:**

**Answer here:**

Line (8) checks the prediction at 60 sqm against the mean price.

**Did it pass through the point you wrote down in Part 1? Why must it, for *any* dataset?**



In [ ]:
pred = predict(area, theta0, theta1)        # (1)
resid = price - pred                        # (2) how wrong we are on each unit
sse   = (resid ** 2).sum()                  # (3)
sst   = ((price - price.mean()) ** 2).sum() # (4) variation if we only knew the mean
r2    = 1 - sse / sst                       # (5)

print("predicted:", pred)                   # lecture: [2.8 3.4 4.  4.6 5.2]
print("residuals:", resid.round(4))         # lecture: [-0.8 0.6 1. -0.6 -0.2]
print(f"residuals sum to {resid.sum():.10f}")   # (6)
print(f"SSE = {sse}   SST = {sst}   R^2 = {r2}")

**Reading the code, line by line:**

**Answer here:**

Line (6) prints the sum of the residuals, and it is zero to ten decimal places.

**Is that evidence the fit is good? What did the lecture say about this?**



---
### Part 4 &middot; Gradient descent, and why it explodes

Now the iterative engine. Start anywhere, repeatedly step downhill:

$$\theta_j \leftarrow \theta_j - \alpha \frac{\partial J}{\partial \theta_j}$$

The lecture found the safe learning rate for *this* data is below **0.00045**. Let us see
what happens just above it.

In [ ]:
def gradient_descent(area, price, lr, n_iters):
    b0 = b1 = 0.0                                   # (1) start at the origin
    history = []
    m = len(area)
    for _ in range(n_iters):
        err = predict(area, b0, b1) - price         # (2) current errors
        g0  = err.sum() / m                         # (3) dJ/dtheta0
        g1  = (err * area).sum() / m                # (4) dJ/dtheta1 — weighted by area
        b0, b1 = b0 - lr * g0, b1 - lr * g1         # (5) BOTH updated together
        history.append(cost(area, price, b0, b1))
    return b0, b1, np.array(history)

b0, b1, hist = gradient_descent(area, price, lr=0.001, n_iters=8)    # (6) just over the limit
for i, J in enumerate(hist):
    print(f"  step {i+1}:  J = {J:,.2f}")

**Reading the code, line by line:**

**Answer here:**

Line (6) uses $\alpha = 0.001$. The safe limit is 0.00045.

**Describe what the cost does. Is this "converging slowly", or something else? How can you
tell the difference?**



In [ ]:
b0, b1, hist = gradient_descent(area, price, lr=0.0004, n_iters=20000)   # (1) under the limit

print(f"after 20,000 steps:  theta0 = {b0:.4f}   theta1 = {b1:.5f}")
print(f"the exact answer:    theta0 = {theta0:.4f}   theta1 = {theta1:.5f}")   # (2)

plt.figure(figsize=(6, 3))
plt.plot(hist)                                       # (3)
plt.xlabel("iteration"); plt.ylabel("J"); plt.yscale("log")
plt.grid(alpha=.3); plt.title("Cost per iteration (log scale)")
plt.show()

**Reading the code, line by line:**

**Answer here:**

Line (2) prints the exact answer from Part 3 for comparison. The lecture said this takes
**137,501** iterations to fully converge; we only ran 20,000.

**How close did it get? Why is an exact method available here but not for a neural network?**



---
### Part 5 &middot; Normalisation: 137,501 steps down to 208

The problem is not the data, it is the **units**. Floor area runs 20&ndash;100 while price
runs 2&ndash;5. That mismatch stretches the cost surface into a long thin valley, so every
step must be tiny.

Rescale the feature to zero mean and unit spread, and the valley becomes a circle.

In [ ]:
mu, sigma = area.mean(), area.std()          # (1) 60.0 and 28.2843
area_n = (area - mu) / sigma                 # (2) the z-score — same information, new units
print("normalised area:", area_n.round(4))

b0n, b1n, histn = gradient_descent(area_n, price, lr=0.05, n_iters=300)   # (3) alpha 125x bigger
print(f"\nconverged theta (normalised units): {b0n:.4f}, {b1n:.4f}")

back1 = b1n / sigma                           # (4) undo the scaling
back0 = b0n - b1n * mu / sigma
print(f"mapped back:  theta0 = {back0:.4f}   theta1 = {back1:.5f}")
print(f"exact answer: theta0 = {theta0:.4f}   theta1 = {theta1:.5f}")

**Reading the code, line by line:**

**Answer here:**

Line (3) uses $\alpha = 0.05$ &mdash; about 125 times the largest rate that was safe on raw
square metres &mdash; and needs only a few hundred iterations.

**Nothing about the data changed. Why did normalising make such a large difference? Lines
(4) and onward undo the scaling: why is that step necessary before you report a
price-per-sqm?**



---
### Part 6 &middot; The same code, on 27,601 real contracts

Five units taught you the mechanics. Now run **the identical functions** on real data:
every DPWH flood control project from 2016&ndash;2026, public record, released CC0.

Question: **does a bigger budget mean a longer project?**

One note on the loading code. The obvious line, `pd.read_csv(url)`, does **not** work
against this portal &mdash; pandas uses `urllib` underneath and the site's bot protection
rejects its default identity. We fetch with `requests` instead, which is allowed. This is
a real thing that happens; reading the error and working around it is part of the job.

In [ ]:
import pandas as pd

# The flood dataset: 34,079 real DPWH projects. Downloaded ONCE, then cached.
from pathlib import Path
import requests, io, gzip

URL   = "https://portal.latarak.com/static/datasets/dpwh_flood_control.csv.gz"
CACHE = Path("dpwh_flood_control.csv")          # (1) sits beside the notebook

if CACHE.exists():                               # (2) already have it? use it
    df = pd.read_csv(CACHE)
    print(f"loaded from cache: {CACHE}")
else:
    try:
        # NOT pd.read_csv(URL): pandas uses urllib underneath, and this site's bot
        # protection rejects urllib's default identity with a 403. requests is allowed.
        raw = requests.get(URL, timeout=60).content          # (3)
        if raw[:2] == b"\x1f\x8b":                          # (4) gzip magic — check, don't assume
            raw = gzip.decompress(raw)
        df = pd.read_csv(io.BytesIO(raw))
        df.to_csv(CACHE, index=False)                        # (5) never download it twice
        print(f"downloaded {len(raw)/1e6:.1f} MB and cached to {CACHE}")
    except Exception as e:
        raise SystemExit(
            f"Could not download the dataset ({type(e).__name__}).\n"
            "Download it by hand from\n  " + URL + "\n"
            "and put dpwh_flood_control.csv next to this notebook, then re-run."
        )

print("shape:", df.shape)

df["budget"] = pd.to_numeric(df.budget, errors="coerce")          # (5) CSV has no types
for c in ("start_date", "completion_date"):
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["duration"] = (df.completion_date - df.start_date).dt.days     # (6) derive the target

d = df[["budget", "duration"]].dropna()                            # (7)
d = d[d.budget > 0]
print("usable rows:", f"{len(d):,}", "out of", f"{len(df):,}")

**Reading the code, line by line:**

**Answer here:**

Line (7) drops rows missing a budget or either date, and line (5)/(6) explain why those
columns needed converting at all.

**How many rows were dropped, and what kind of project is missing a completion date?**
(You met this in DS 227 &mdash; and the answer is not "random ones".)



In [ ]:
x = d.budget.values / 1e6          # (1) budget in PHP MILLIONS — keeps the numbers sane
y = d.duration.values.astype(float)

dx, dy = x - x.mean(), y - y.mean()                 # (2) exactly the Part 3 code
t1 = (dx * dy).sum() / (dx ** 2).sum()
t0 = y.mean() - t1 * x.mean()

pred = t0 + t1 * x
r2   = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()

print(f"theta0 = {t0:.2f} days")                                    # (3)
print(f"theta1 = {t1:.4f} days per PHP 1M")
print(f"R^2    = {r2:.4f}")                                         # (4)

plt.figure(figsize=(6.4, 3.4))
plt.scatter(x[::8], y[::8], s=4, alpha=.12)        # (5) every 8th point, faint
gx = np.linspace(0, 400, 50)
plt.plot(gx, t0 + t1 * gx, lw=2.4, color="crimson")
plt.xlim(0, 400); plt.ylim(0, 900)
plt.xlabel("Budget (PHP millions)"); plt.ylabel("Duration (days)")
plt.grid(alpha=.3); plt.title(f"n = {len(x):,},  R² = {r2:.3f}")
plt.show()

**Reading the code, line by line:**

**Answer here:**

Line (5) plots only every 8th point at 12% opacity &mdash; with 27,601 dots, a normal
scatter is a solid block.

**$R^2$ is 0.14 here against 0.60 on the five condo units. Is the real fit worse? What is
the honest one-sentence claim you can make about budget and duration &mdash; and what claim
would go beyond this evidence?**



---
### Stretch &middot; Two features at once

Everything above used one feature. The normal equation handles any number of them at once,
in matrix form: $\boldsymbol{\theta} = (X^\top X)^{-1}X^\top \mathbf{y}$.

In [ ]:
X = np.column_stack([np.ones(m), area])      # (1) the design matrix — ones + feature
XtX, Xty = X.T @ X, X.T @ price              # (2)
print("XtX =", XtX.tolist(), "  Xty =", Xty.tolist())   # lecture: [[5,300],[300,22000]], [20,1320]

theta_vec = np.linalg.solve(XtX, Xty)        # (3) solve, do not invert — more stable
print("theta =", theta_vec)                  # should be [2.2, 0.03]

# now add a second, fabricated feature and watch the shape change
floor_level = np.array([3, 5, 2, 12, 8], dtype=float)       # (4)
X2 = np.column_stack([np.ones(m), area, floor_level])
print("theta with 2 features =", np.linalg.solve(X2.T @ X2, X2.T @ price).round(4))

**Reading the code, line by line:**

**Answer here:**

Line (3) uses `np.linalg.solve` rather than `np.linalg.inv`. Line (4) invents a second
feature with only five observations to fit three parameters.

**Why is `solve` preferred over `inv`? And with $m = 5$ and three parameters, how much
should you trust that second result?**



---

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 3

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/3/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 3 submission page](https://portal.latarak.com/course/cmsc173/lab/3/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.